# SuperMockLoad — quickstart

Load a SuperMock skypatch, look at the basics, and make a couple of plots.
The package ships the observational comparison data; point it at the synthetic
catalogs via `SUPERMOCK_DATA` or `root=`.

In [ ]:
import os
# Set this to wherever the SuperMock patch files live:
os.environ['SUPERMOCK_DATA'] = '/lcrc/project/cosmo_ai/nramachandra/Projects/SPHEREx/Mocks_v3_data'

In [ ]:
%matplotlib inline
import numpy as np, matplotlib.pyplot as plt
from supermockload import SuperMock, available_patches, plots, observations

In [ ]:
available_patches()

## The simulation & how galaxies are made

SuperMock galaxies are painted onto the **HACC Last Journey** gravity-only
N-body simulation — a 5025 Mpc/h box (~1.24 trillion particles) in a
Planck-like cosmology (H0 = 67.66, Ωm = 0.310). Dark-matter *cores* are
tracked across **101 analysis snapshots** from z ≈ 10 to 0.

Each core is matched to a **UniverseMachine** galaxy (on the SMDPL
simulation) by its assembly history — a nearest-neighbour match on
`[t50, t25, peak_mass, infall_time]` using the core's *own* mass track — which
supplies a star-formation history and stellar mass. **FSPS** (C3K stellar
libraries) with a calibrated two-component dust model turns each SFH into a
rest-frame SED, and the SEDs are projected through the real survey filters to
give photometry.

One simulation / lightcone is available today; more skypatches (eventually the
full sky) share the same schema and combine with `SuperMock([...])`.

### Load one patch
`downsample=` keeps a random subset (fast, low memory). Drop it for all rows.

In [ ]:
sm = SuperMock(3, downsample=300_000)
sm

In [ ]:
print('footprint  :', round(sm.area_deg2, 1), 'deg^2')
print('N galaxies :', f'{sm.n:,}  (downsampled)')
print('z median   :', round(np.median(sm.redshift), 3))
print('logM* median (obs-epoch):', round(np.median(sm.logM_zobs), 2))

## Catalog entries

Per-galaxy fields in the **catalog** file (`sm.field(...)`, or scalars by
attribute like `sm.redshift`):

| field | meaning |
|---|---|
| `ra`, `dec` | sky position [deg] on the lightcone |
| `x, y, z` | comoving position [Mpc/h]; `vx, vy, vz` peculiar velocity [km/s] |
| `redshift` | observed redshift (cosmological + peculiar) |
| `stellar_mass` | matched-UM stellar mass at a=1 [M⊙] |
| `stellar_mass_zobs` | **observed-epoch** stellar mass (SFH integrated to z_obs) [M⊙] — use this for mass functions |
| `peak_mass` | peak halo mass [M⊙/h]; `rank_peak_mass` its rank (matching) |
| `sfh` | (117) star-formation history SFR(t) [M⊙/yr] on `sm.sfh_time` |
| `mah`, `mah_host` | (101) halo / host-halo mass M(t) [M⊙/h] on `sm.mah_time` |
| `core_state_history` | (101) core state per snapshot (-1 not formed, 0 central, 1 satellite, 2 merged) |
| `central` | 1 if central of its FoF halo |
| `merged` | 1 if the core has merged (terminal); `core_state` its state at z_obs |
| `core_tag` | unique core id; `fof_halo_tag` groups galaxies into halos |
| `t50_a1`, `t25_a1` | snapshot index where the halo reached 50% / 25% of peak mass |
| `time_infall` | infall snapshot index |

Derived quantities live in **separate row-aligned files** (same row order as
the catalog):

| quantity | file / accessor | shape | meaning |
|---|---|---|---|
| photometry | `sm.mag(survey,band)`, `sm.survey(survey)` | (N, n_band) | apparent AB mag per band |
| — surveys | `LSST`(6) `WISE`(7) `SPHEREx`(102) `COSMOS`(31) `LEGACYSURVEY`(8) `2MASS`(3) `F784`(2) | | `sm.bands(survey)` for labels, `sm.wavelengths(survey)` for λ_eff |
| `M_ABS_SDSS` | `sm.abs_mag('SDSS',b)` | (N, 5) | rest-frame absolute mag, ugriz |
| `M_ABS_WISE` | `sm.abs_mag('WISE',b)` | (N, 4) | rest-frame absolute mag, W1–W4 |
| `L_BOL_LSUN` | `sm.luminosity('L_BOL_LSUN')` | (N,) | bolometric luminosity [L⊙] |
| `L_8_33UM_LSUN` | `sm.luminosity('L_8_33UM_LSUN')` | (N,) | 8–33 μm luminosity [L⊙] |
| SED | `sm.seds(rows=...)` | (n, 11149) | `f_ν` [Jy] on the rest grid `wave_rest` [Å]; observed λ = wave_rest·(1+z) |

The **filter response curves** used to convolve the SEDs into photometry ship
with the package: `sm.bandpass(survey, band)` → `(λ[μm], transmission)`, or
`sm.bandpass(survey)` for all bands; `plots.bandpasses(sm)` overlays them.

In [ ]:
# the actual bandpasses used for the photometry
w, t = sm.bandpass('LSST', 'i')                 # (wavelength[um], transmission)
print('LSST i bandpass:', w.shape, 'peak at %.2f um' % w[t.argmax()])
allsx = sm.bandpass('SPHEREx')                  # dict: 102 channels
print('SPHEREx channels:', len(allsx))
fig, ax = plt.subplots(figsize=(9, 3.2)); plots.bandpasses(sm, ax=ax); fig.tight_layout()

### Accessing entries — desired and all available

In [ ]:
sm.catalog_fields()          # EVERY catalog field on disk: name -> (shape, dtype)

In [ ]:
# scalars by attribute; any field via .field(); 2-D fields need extra_fields=
z   = sm.redshift                        # == sm.field('redshift')
lm  = sm.logM_zobs                       # log10 observed-epoch stellar mass
print('loaded so far:', sm.loaded_fields())

# photometry / luminosities (loaded lazily on first use)
mi  = sm.mag('LSST', 'i')                # one band, AB
spx = sm.survey('SPHEREx')               # full (N, 102); labels: sm.bands('SPHEREx')
Mr  = sm.abs_mag('SDSS', 'r')            # rest-frame absolute mag
Lb  = sm.luminosity('L_BOL_LSUN')

# the x-axes for the histories / spectra (part of the package)
print('MAH time  :', sm.mah_time[[0, -1]], 'Gyr   (101 snapshots)')
print('SFH time  :', sm.sfh_time[[0, -1]], 'Gyr   (117 bins)')
print('SPHEREx wl:', sm.wavelengths('SPHEREx')[[0, -1]], 'micron   (102 channels)')
print('SED grid  : sm.seds(rows=...) returns (wave_rest[A], f_nu[Jy])')

The 2-D histories (`sfh`, `mah`) are only read when asked for:

In [ ]:
sm_h = SuperMock(3, downsample=50_000, extra_fields=('sfh', 'mah'))
print('sfh:', sm_h.field('sfh').shape, 'on sm_h.sfh_time', sm_h.sfh_time.shape)
print('mah:', sm_h.field('mah').shape, 'on sm_h.mah_time', sm_h.mah_time.shape)

### A couple of plots vs observations

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
plots.redshift_distribution(sm, ax=ax[0])
plots.gsmf(sm, ax=ax[1])
plots.number_counts(sm, ax=ax[2], survey='LSST', band='i')
fig.tight_layout()

### Fast repeat loads: snapshots

The catalog/photometry files are gzip-compressed, so the first read is slow.
Snapshot a downsampled view to an uncompressed file that reloads instantly.

In [ ]:
rows = sm.sample(2000)                                  # for a stored SED subset
# needs seds=True to store SEDs; here we snapshot catalog+photometry only:
sm.save('patch3_300k.snapshot.h5', surveys=('LSST', 'WISE'))
sm2 = SuperMock.from_file('patch3_300k.snapshot.h5')    # instant
sm2